In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.models as models
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!apt-get install unrar

In [ ]:
!unrar x /content/drive/MyDrive/Dataset.rar /content/

In [ ]:
class faces_dataset(datasets.ImageFolder):
  def __init__(self,img_dir,transform=None):
    super().__init__(img_dir,transform=transform)
    self.class_idx={'faces': 0.0, 'non-faces': 1.0}
  def __getitem__(self, idx):
    img_path,_=self.samples[idx]
    class_name=os.path.basename(os.path.dirname(img_path))

    img_label=self.class_idx[class_name]
    image= self.loader(img_path)
    if self.transform is not None:
      image= self.transform(image)
    return image,img_label

In [ ]:
"""*********"""
import torch.nn.functional as F
class modified_VGG16(nn.Module):
  def __init__(self):
    super().__init__()
    self.backbone=models.vgg16(pretrained=True).features
    self.new_fc_layer=nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.3,inplace=False),
    nn.Linear(256, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.3,inplace=False),
    nn.Linear(128, 1),
    nn.Sigmoid() )
    for param in self.backbone.parameters():
      param.requires_gards = False

  def forward(self, x):
        x = self.backbone(x)
        x = F.max_pool2d(x,kernel_size=x.size()[2:])
        x = x.view(x.size(0), -1)
        x = self.new_fc_layer(x)
        x=x.squeeze(1)

        return x

In [ ]:
modified_VGG16()

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 233MB/s]


modified_VGG16(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0,

In [ ]:
def train(dataloader, model, loss_fn, optimizer,current_epoch):
    size = len(dataloader.dataset)
    correct = 0

    model.train()
    for batch, (X,label) in enumerate(dataloader):

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        X = X.to(device)
        label = label.to(device).float()

        Y = model(X).float()
        loss = loss_fn(Y,label)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


        predicted = torch.round(Y)
        correct += (predicted == label).sum().item()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


    train_acc = 100 * correct / size
    print(f"Training Accuracy: {train_acc:.2f}%")

    path = f"drive/MyDrive/model_plates/model_epoch_{current_epoch}.pt"
    torch.save(model.state_dict(), path)

In [ ]:
def test(dataloader, model, loss_fn):

    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for X,label in dataloader:

            device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
            X = X.to(device)
            label = label.to(device).float()

            Y = model(X).float()
            test_loss += loss_fn(Y,label).item()
            correct += (torch.round(Y) == label).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [ ]:
faces_data = faces_dataset('/content/Dataset',transform)

train_size = int(0.75 * len( faces_data))
test_size = len( faces_data ) - train_size
training_faces_data, testing_faces_data = data.random_split( faces_data , [train_size, test_size])

train_dataloader_faces=data.DataLoader(training_faces_data,batch_size=64,shuffle=True)
test_dataloader_faces=data.DataLoader(testing_faces_data,batch_size=64,shuffle=True)

In [ ]:
faces_model = modified_VGG16()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
faces_model = faces_model.to(device)


optimizer = optim.Adam = (faces_model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

if not os.path.exists('drive/MyDrive/models'):
    os.makedirs('drive/MyDrive/models')

epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader_faces, faces_model, loss_fn, optimizer,t+1)
    test(test_dataloader_faces, faces_model, loss_fn)


print("Done!")